In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import lightgbm as lgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau

from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "/kaggle/input/datasets/eskobar/msme-dataset/umkm_success_prediction_with_success.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

2026-06-03 15:31:01.439578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780500661.598221      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780500661.651147      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780500662.068233      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780500662.068298      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780500662.068304      22 computation_placer.cc:177] computation placer alr

# EDA

In [2]:
def run_eda(df: pd.DataFrame) -> None:
    print("EDA")

    print(f"\nShape         : {df.shape}")
    print(f"Duplicates    : {df.duplicated().sum()}")
    print(f"Missing values:\n{df.isnull().sum()}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nDesc:\n{df.describe().T.to_string()}")

    vc = df["Success"].value_counts()
    print(f"\nTarget distribution:\n{vc}")
    print(f"  Imbalance ratio (major/minor): {vc[0]/vc[1]:.2f}x")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    vc.plot(kind="bar", ax=axes[0], color=["#e74c3c", "#2ecc71"], edgecolor="black")
    axes[0].set_title("Target Class Distribution")
    axes[0].set_xlabel("Success")
    axes[0].set_ylabel("Count")
    axes[0].set_xticklabels(["Fail (0)", "Success (1)"], rotation=0)
    for p in axes[0].patches:
        axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + 0.15, p.get_height() + 1))

    corr = df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
                annot=True, fmt=".2f", linewidths=0.5, ax=axes[1])
    axes[1].set_title("Feature Correlation Matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eda_target_and_correlation.png", dpi=150)
    plt.close()

    features = [c for c in df.columns if c != "Success"]
    cols = 4
    rows = (len(features) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes = axes.flatten()

    for i, feat in enumerate(features):
        for label, color in zip([0, 1], ["#e74c3c", "#2ecc71"]):
            subset = df[df["Success"] == label][feat]
            axes[i].hist(subset, bins=15, alpha=0.6, color=color,
                         label=f"Class {label}", edgecolor="black", linewidth=0.5)
        axes[i].set_title(feat, fontsize=9)
        axes[i].legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Feature Distributions by Target Class", y=1.01, fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eda_feature_distributions.png", dpi=150, bbox_inches="tight")
    plt.close()

    print("\nEDA plots saved to outputs/")

# DATA PREPROCESSING

In [3]:
def preprocess(df: pd.DataFrame):
    print("DATA PREPROCESSING")

    X = df.drop(columns=["Success"])
    y = df["Success"]

    FEATURE_NAMES = X.columns.tolist()
    print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=SEED, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.15 / 0.85,
        random_state=SEED, stratify=y_train_val
    )

    print(f"\ntrain={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
    print(f"Train: {Counter(y_train)}")
    print(f"Val: {Counter(y_val)}")
    print(f"Test: {Counter(y_test)}")

    # Oversampling train for handling imbalance target class (success vs fail)
    sm = SMOTE(random_state=SEED, k_neighbors=5)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f"\nAfter SMOTE  : {Counter(y_train_res)}")

    # Use RobustScaler, more robust for handling outlier in dataset
    scaler = RobustScaler()
    X_train_sc  = scaler.fit_transform(X_train_res)
    X_val_sc    = scaler.transform(X_val)
    X_test_sc   = scaler.transform(X_test)

    data = {
        "X_train_raw": X_train_res, "y_train": y_train_res,
        "X_val_raw"  : X_val,       "y_val"  : y_val,
        "X_test_raw" : X_test,      "y_test" : y_test,
        "X_train_sc" : X_train_sc,
        "X_val_sc"   : X_val_sc,
        "X_test_sc"  : X_test_sc,
        "scaler"     : scaler,
        "feature_names": FEATURE_NAMES,
    }
    return data

# ML MODELS
##  (Random Forest, XGBoost, LightGBM, Gradient Boosting)

In [4]:
def evaluate_model(name, model, X_val, y_val, X_test, y_test):
    print(f"  {name}")

    y_val_pred  = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_val_prob  = model.predict_proba(X_val)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_val_prob  = model.decision_function(X_val)
        y_test_prob = model.decision_function(X_test)
    else:
        y_val_prob = y_val_pred.astype(float)
        y_test_prob = y_test_pred.astype(float)

    print("\n[Validation set]")
    print(classification_report(y_val, y_val_pred, target_names=["Fail", "Success"]))
    print(f"  ROC-AUC: {roc_auc_score(y_val, y_val_prob):.4f}")
    print(f"  PR-AUC : {average_precision_score(y_val, y_val_prob):.4f}")

    print("\n[Test set]")
    print(classification_report(y_test, y_test_pred, target_names=["Fail", "Success"]))
    print(f"  ROC-AUC: {roc_auc_score(y_test, y_test_prob):.4f}")
    print(f"  PR-AUC : {average_precision_score(y_test, y_test_prob):.4f}")

    return {
        "name"         : name,
        "val_roc_auc"  : roc_auc_score(y_val, y_val_prob),
        "test_roc_auc" : roc_auc_score(y_test, y_test_prob),
        "test_pr_auc"  : average_precision_score(y_test, y_test_prob),
        "y_test_pred"  : y_test_pred,
        "y_test_prob"  : y_test_prob,
    }


def train_ml_models(data: dict) -> list:
    print("ML MODELS")

    X_tr = data["X_train_raw"]
    y_tr = data["y_train"]
    X_v  = data["X_val_raw"]
    y_v  = data["y_val"]
    X_te = data["X_test_raw"]
    y_te = data["y_test"]

    class_weight = {0: 1.0, 1: len(y_tr[y_tr == 0]) / len(y_tr[y_tr == 1])}
    scale_pos_weight = class_weight[1]   # ratio fail vs success

    models = {

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            oob_score=True,
            n_jobs=-1,
            random_state=SEED,
        ),

        "XGBoost": XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            use_label_encoder=False,
            random_state=SEED,
            n_jobs=-1,
        ),

        "LightGBM": lgb.LGBMClassifier(
            n_estimators=400,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=10,
            reg_alpha=0.1,
            reg_lambda=1.0,
            is_unbalance=True,  
            random_state=SEED,
            n_jobs=-1,
            verbose=-1,
        ),

        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            min_samples_leaf=4,
            random_state=SEED,
        ),
    }

    print("\n--- Five-Fold Cross-Validation (on SMOTE-augmented train set) ---")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    for name, model in models.items():
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=skf,
                                    scoring="roc_auc", n_jobs=-1)
        print(f"  {name:25s}: CV ROC-AUC = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    results = []
    fitted_models = {}

    for name, model in models.items():
        if name == "XGBoost":
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_v, y_v)],
                verbose=False,
            )
        elif name == "LightGBM":
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_v, y_v)],
                callbacks=[lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(period=-1)],
            )
        else:
            model.fit(X_tr, y_tr)

        fitted_models[name] = model
        res = evaluate_model(name, model, X_v, y_v, X_te, y_te)
        results.append(res)

    fn = data["feature_names"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, mname in zip(axes, ["Random Forest", "XGBoost"]):
        m = fitted_models[mname]
        imp = m.feature_importances_
        idx = np.argsort(imp)
        ax.barh([fn[i] for i in idx], imp[idx], color="#3498db", edgecolor="black")
        ax.set_title(f"{mname}\nFeature Importance")
        ax.set_xlabel("Importance")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ml_feature_importance.png", dpi=150)
    plt.close()

    return results, fitted_models

# DL Functional API + Custom Callbacks

In [5]:
class MetricLogger(Callback):
    def on_train_begin(self, logs=None):
        self.history = {
            "epoch": [], "loss": [], "val_loss": [],
            "auc": [], "val_auc": [],
            "precision": [], "val_precision": [],
            "recall": [], "val_recall": [],
        }
        print("\n  [Logger] Training started will log every epoch")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.history["epoch"].append(epoch + 1)
        self.history["loss"].append(logs.get("loss"))
        self.history["val_loss"].append(logs.get("val_loss"))
        self.history["auc"].append(logs.get("auc"))
        self.history["val_auc"].append(logs.get("val_auc"))
        self.history["precision"].append(logs.get("precision"))
        self.history["val_precision"].append(logs.get("val_precision"))
        self.history["recall"].append(logs.get("recall"))
        self.history["val_recall"].append(logs.get("val_recall"))

    def on_train_end(self, logs=None):
        best_epoch = np.argmax(self.history["val_auc"]) + 1
        best_auc   = max(self.history["val_auc"])
        print(f"  [Logger] Training done. "
              f"Best val_auc={best_auc:.4f} at epoch {best_epoch}.")

    def plot(self, save_path: Path):
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        epochs = self.history["epoch"]

        for ax, (train_key, val_key, title) in zip(axes, [
            ("loss",      "val_loss",      "Loss"),
            ("auc",       "val_auc",       "ROC-AUC"),
            ("precision", "val_precision", "Precision & Recall"),
        ]):
            ax.plot(epochs, self.history[train_key], label=f"Train {title}", marker=".")
            ax.plot(epochs, self.history[val_key],   label=f"Val   {title}", marker=".")
            if train_key == "precision":
                ax.plot(epochs, self.history["recall"],     label="Train Recall", linestyle="--")
                ax.plot(epochs, self.history["val_recall"], label="Val   Recall", linestyle="--")
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.close()


class F1ScoreCallback(Callback):
    """Computes macro-F1 on the validation set at each epoch end.
       Saves the best model weights based on val_f1."""

    def __init__(self, val_data, save_path: str):
        super().__init__()
        self.X_val, self.y_val = val_data
        self.save_path = save_path
        self.best_f1   = -np.inf

    def on_epoch_end(self, epoch, logs=None):
        from sklearn.metrics import f1_score
        y_pred = (self.model.predict(self.X_val, verbose=0) >= 0.5).astype(int).flatten()
        f1 = f1_score(self.y_val, y_pred, average="macro", zero_division=0)

        if f1 > self.best_f1:
            self.best_f1 = f1
            self.model.save_weights(self.save_path)
            print(f"  [F1Callback] Epoch {epoch+1:03d} | val_macro_f1={f1:.4f}  ✓ Saved best weights")
        else:
            print(f"  [F1Callback] Epoch {epoch+1:03d} | val_macro_f1={f1:.4f}")


class LearningRateMonitor(Callback):
    """Prints the current learning rate at the end of each epoch."""

    def on_epoch_end(self, epoch, logs=None):
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        print(f"  [LRMonitor]  Epoch {epoch+1:03d} | lr = {lr:.2e}")


class GradientNormMonitor(Callback):
    """Logs the global gradient L2 norm every N steps to detect exploding/vanishing gradients."""

    def __init__(self, log_every: int = 50):
        super().__init__()
        self.log_every   = log_every
        self.grad_norms  = []
        self._step       = 0

    def on_train_batch_end(self, batch, logs=None):
        self._step += 1
        if self._step % self.log_every == 0:
            # Retrieve the last recorded gradient norm if available
            # (TF graph-mode: we approximate via weights delta or use tf.GradientTape in a custom training loop)
            pass  # Placeholder — see note below in build_dl_model

    def summary(self):
        if self.grad_norms:
            print(f"  [GradNorm] Mean={np.mean(self.grad_norms):.4f} "
                  f"Max={np.max(self.grad_norms):.4f}")


class ThresholdTuner(Callback):
    """After training ends, sweeps decision thresholds on val set to find
       the one that maximises macro-F1, then prints results."""

    def __init__(self, val_data, thresholds=None):
        super().__init__()
        self.X_val, self.y_val = val_data
        self.thresholds = thresholds or np.arange(0.1, 0.91, 0.05)
        self.best_threshold = 0.5
        self.best_f1 = -np.inf

    def on_train_end(self, logs=None):
        from sklearn.metrics import f1_score
        probs = self.model.predict(self.X_val, verbose=0).flatten()
        print("\n  [ThresholdTuner] Sweeping decision thresholds on val set:")

        for t in self.thresholds:
            preds = (probs >= t).astype(int)
            f1 = f1_score(self.y_val, preds, average="macro", zero_division=0)
            if f1 > self.best_f1:
                self.best_f1        = f1
                self.best_threshold = t

        print(f"  [ThresholdTuner] Best threshold = {self.best_threshold:.2f} "
              f"→ val macro-F1 = {self.best_f1:.4f}")


def build_dl_model(input_dim: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name="features")

    x = layers.BatchNormalization(name="bn_input")(inputs)
    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_1"
    )(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dropout(0.4, name="drop_1")(x)

    x2 = layers.Dense(
        64, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_2"
    )(x)
    x2 = layers.BatchNormalization(name="bn_2")(x2)
    x2 = layers.Dropout(0.3, name="drop_2")(x2)

    shortcut = layers.Dense(64, use_bias=False, name="shortcut")(x)
    x3 = layers.Add(name="residual_add")([x2, shortcut])
    x3 = layers.Activation("relu", name="residual_relu")(x3)

    x4 = layers.Dense(
        32, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_3"
    )(x3)
    x4 = layers.Dropout(0.2, name="drop_3")(x4)

    output = layers.Dense(1, activation="sigmoid", name="output")(x4)

    model = keras.Model(inputs=inputs, outputs=output, name="UMKM_DL_Classifier")
    return model


def train_dl_model(data: dict) -> dict:
    print("\n" + "=" * 70)
    print("4. DEEP LEARNING (Functional API + Custom Callbacks)")
    print("=" * 70)

    X_tr = data["X_train_sc"].astype("float32")
    y_tr = data["y_train"].values.astype("float32")
    X_v  = data["X_val_sc"].astype("float32")
    y_v  = data["y_val"].values.astype("float32")
    X_te = data["X_test_sc"].astype("float32")
    y_te = data["y_test"].values.astype("float32")

    input_dim = X_tr.shape[1]

    pos   = int(y_tr.sum())
    neg   = len(y_tr) - pos
    cw    = {0: 1.0, 1: neg / pos}
    print(f"\nClass weights: {cw}")

    model = build_dl_model(input_dim)
    model.summary()

    BEST_WEIGHTS_PATH = str(OUTPUT_DIR / "dl_best_weights_f1.weights.h5")

    optimizer = keras.optimizers.Adam(learning_rate=1e-3)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

    metric_logger     = MetricLogger()
    f1_saver          = F1ScoreCallback(val_data=(X_v, y_v), save_path=BEST_WEIGHTS_PATH)
    lr_monitor        = LearningRateMonitor()
    grad_monitor      = GradientNormMonitor(log_every=50)
    threshold_tuner   = ThresholdTuner(val_data=(X_v, y_v))

    early_stop  = EarlyStopping(
        monitor="val_auc", mode="max", patience=30,
        restore_best_weights=True, verbose=1
    )
    reduce_lr   = ReduceLROnPlateau(
        monitor="val_auc", mode="max", factor=0.5,
        patience=10, min_lr=1e-6, verbose=1
    )

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=200,
        batch_size=32,
        class_weight=cw,
        callbacks=[
            metric_logger,
            f1_saver,
            lr_monitor,
            grad_monitor,
            threshold_tuner,
            early_stop,
            reduce_lr,
        ],
        verbose=0,    
    )

    if Path(BEST_WEIGHTS_PATH).exists():
        model.load_weights(BEST_WEIGHTS_PATH)
        print("\n[DL] Loaded best weights (by val macro-F1).")

    best_threshold = threshold_tuner.best_threshold

    print(f"\n--- DL eval (threshold = {best_threshold:.2f}) ---")

    y_val_prob  = model.predict(X_v, verbose=0).flatten()
    y_test_prob = model.predict(X_te, verbose=0).flatten()

    y_val_pred  = (y_val_prob  >= best_threshold).astype(int)
    y_test_pred = (y_test_prob >= best_threshold).astype(int)

    print("\n[Validation set]")
    print(classification_report(y_v, y_val_pred, target_names=["Fail", "Success"]))
    print(f"  ROC-AUC: {roc_auc_score(y_v, y_val_prob):.4f}")
    print(f"  PR-AUC : {average_precision_score(y_v, y_val_prob):.4f}")

    print("\n[Test set]")
    print(classification_report(y_te, y_test_pred, target_names=["Fail", "Success"]))
    print(f"  ROC-AUC: {roc_auc_score(y_te, y_test_prob):.4f}")
    print(f"  PR-AUC : {average_precision_score(y_te, y_test_prob):.4f}")

    metric_logger.plot(OUTPUT_DIR / "dl_training_curves.png")
    model.save(OUTPUT_DIR / "dl_umkm_model.keras")

    dl_result = {
        "name"        : "Deep Learning",
        "val_roc_auc" : roc_auc_score(y_v, y_val_prob),
        "test_roc_auc": roc_auc_score(y_te, y_test_prob),
        "test_pr_auc" : average_precision_score(y_te, y_test_prob),
        "y_test_pred" : y_test_pred,
        "y_test_prob" : y_test_prob,
        "threshold"   : best_threshold,
    }

    return dl_result, model

# Evaluation

In [6]:
def final_evaluation(all_results: list, y_test):
    print("FINAL EVALUATION SUMMARY")

    df_res = pd.DataFrame([{
        "Model"        : r["name"],
        "Val ROC-AUC"  : r["val_roc_auc"],
        "Test ROC-AUC" : r["test_roc_auc"],
        "Test PR-AUC"  : r["test_pr_auc"],
    } for r in all_results]).sort_values("Test ROC-AUC", ascending=False).reset_index(drop=True)

    print("\n", df_res.to_string(index=False))

    # ROC plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for r in all_results:
        fpr, tpr, _ = roc_curve(y_test, r["y_test_prob"])
        axes[0].plot(fpr, tpr, lw=1.8,
                     label=f"{r['name']} (AUC={r['test_roc_auc']:.3f})")

    axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
    axes[0].set_xlabel("FPR")
    axes[0].set_ylabel("TPR")
    axes[0].set_title("ROC Curves — Test Set")
    axes[0].legend(fontsize=7, loc="lower right")
    axes[0].grid(True, alpha=0.3)

    # PR plot
    for r in all_results:
        prec, rec, _ = precision_recall_curve(y_test, r["y_test_prob"])
        axes[1].step(rec, prec, where="post", lw=1.8,
                     label=f"{r['name']} (AP={r['test_pr_auc']:.3f})")

    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("PR Curves — Test Set")
    axes[1].legend(fontsize=7)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "final_roc_pr_curves.png", dpi=150)
    plt.close()

    # Confusion Matrix
    n = len(all_results)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = axes.flatten()

    for i, r in enumerate(all_results):
        cm = confusion_matrix(y_test, r["y_test_pred"])
        disp = ConfusionMatrixDisplay(cm, display_labels=["Fail", "Success"])
        disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
        axes[i].set_title(r["name"], fontsize=9)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Confusion Matrix — Test Set", y=1.01, fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "final_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.close()

# Inference

In [7]:
def inference_example(fitted_ml_models: dict, dl_model, scaler, feature_names: list):
    print("INFERENCE SIM")

    # Simulating new UMKM owner with characteristics below
    new_sample = {
        "Age"                       : 35,
        "Education"                 : 3,   
        "Initial_Capital"           : 1,   
        "Financial_Record_Keeping"  : 1,   
        "Internet_Usage"            : 1,   
        "Business_Plan"             : 1,   
        "Marketing_Effort"          : 6,
        "Partnership"               : 1,
        "Parent_Business_Experience": 1,
        "Industry_Experience"       : 8,
        "Owner_Gender"              : 0,
        "Professional_Advice"       : 5,
    }

    X_new_raw = pd.DataFrame([new_sample])[feature_names]
    X_new_sc  = scaler.transform(X_new_raw).astype("float32")

    print(f"\nNew sample features:\n{X_new_raw.to_string(index=False)}\n")

    print("ML Model Predictions:")
    for name, model in fitted_ml_models.items():
        pred  = model.predict(X_new_raw)[0]
        prob  = model.predict_proba(X_new_raw)[0, 1]
        label = " SUCCESS" if pred == 1 else " FAILED"
        print(f"  {name:25s}: {label}  (P(success) = {prob:.4f})")

    dl_prob = float(dl_model.predict(X_new_sc, verbose=0)[0][0])
    dl_pred = int(dl_prob >= 0.5)
    dl_label = " SUCCESS" if dl_pred == 1 else " FAILED"
    print(f"  {'Deep Learning':25s}: {dl_label}  (P(success) = {dl_prob:.4f})")

# Runnnn

In [8]:
print("UMKM SUCCESS PREDICTION — ML + DL PIPELINE")

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

run_eda(df)

data = preprocess(df)

UMKM SUCCESS PREDICTION — ML + DL PIPELINE
Loaded: 250 rows × 13 columns
EDA

Shape         : (250, 13)
Duplicates    : 0
Missing values:
Age                           0
Education                     0
Initial_Capital               0
Financial_Record_Keeping      0
Internet_Usage                0
Business_Plan                 0
Marketing_Effort              0
Partnership                   0
Parent_Business_Experience    0
Industry_Experience           0
Owner_Gender                  0
Professional_Advice           0
Success                       0
dtype: int64

Dtypes:
Age                           int64
Education                     int64
Initial_Capital               int64
Financial_Record_Keeping      int64
Internet_Usage                int64
Business_Plan                 int64
Marketing_Effort              int64
Partnership                   int64
Parent_Business_Experience    int64
Industry_Experience           int64
Owner_Gender                  int64
Professional_Advice         

In [9]:
ml_results, fitted_ml_models = train_ml_models(data)

ML MODELS

--- Five-Fold Cross-Validation (on SMOTE-augmented train set) ---
  Random Forest            : CV ROC-AUC = 0.9701 ± 0.0134


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:31:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:31:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:31:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:31:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

  XGBoost                  : CV ROC-AUC = 0.9618 ± 0.0211
  LightGBM                 : CV ROC-AUC = 0.9719 ± 0.0142
  Gradient Boosting        : CV ROC-AUC = 0.9790 ± 0.0110
  Random Forest

[Validation set]
              precision    recall  f1-score   support

        Fail       0.87      0.93      0.90        29
     Success       0.71      0.56      0.62         9

    accuracy                           0.84        38
   macro avg       0.79      0.74      0.76        38
weighted avg       0.83      0.84      0.83        38

  ROC-AUC: 0.9195
  PR-AUC : 0.6532

[Test set]
              precision    recall  f1-score   support

        Fail       0.85      0.76      0.80        29
     Success       0.42      0.56      0.48         9

    accuracy                           0.71        38
   macro avg       0.63      0.66      0.64        38
weighted avg       0.74      0.71      0.72        38

  ROC-AUC: 0.7854
  PR-AUC : 0.4282
  XGBoost

[Validation set]
              precision   

In [10]:
dl_result, dl_model = train_dl_model(data)


4. DEEP LEARNING (Functional API + Custom Callbacks)

Class weights: {0: 1.0, 1: 1.0}


I0000 00:00:1780500792.290551      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780500792.296692      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "UMKM_DL_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ features            │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_input            │ (None, 12)        │         48 │ features[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      1,664 │ bn_input[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_1 (Dropout)    │ (None, 128)       │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ drop_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 64)        │        256 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_2 (Dropout)    │ (None, 64)        │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shortcut (Dense)    │ (None, 64)        │      8,192 │ drop_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_add (Add)  │ (None, 64)        │          0 │ drop_2[0][0],     │
│                     │                   │            │ shortcut[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_relu       │ (None, 64)        │          0 │ residual_add[0][… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ residual_relu[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_3 (Dropout)    │ (None, 32)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         33 │ drop_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,041 (82.19 KB)

 Trainable params: 20,633 (80.60 KB)

 Non-trainable params: 408 (1.59 KB)


  [Logger] Training started will log every epoch


I0000 00:00:1780500796.578309     248 service.cc:152] XLA service 0x34310640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780500796.578347     248 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780500796.578350     248 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780500797.176369     248 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780500800.257541     248 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  [F1Callback] Epoch 001 | val_macro_f1=0.4242  ✓ Saved best weights
  [LRMonitor]  Epoch 001 | lr = 1.00e-03
  [F1Callback] Epoch 002 | val_macro_f1=0.4062
  [LRMonitor]  Epoch 002 | lr = 1.00e-03
  [F1Callback] Epoch 003 | val_macro_f1=0.4896  ✓ Saved best weights
  [LRMonitor]  Epoch 003 | lr = 1.00e-03
  [F1Callback] Epoch 004 | val_macro_f1=0.4896
  [LRMonitor]  Epoch 004 | lr = 1.00e-03
  [F1Callback] Epoch 005 | val_macro_f1=0.5052  ✓ Saved best weights
  [LRMonitor]  Epoch 005 | lr = 1.00e-03
  [F1Callback] Epoch 006 | val_macro_f1=0.5052
  [LRMonitor]  Epoch 006 | lr = 1.00e-03
  [F1Callback] Epoch 007 | val_macro_f1=0.5052
  [LRMonitor]  Epoch 007 | lr = 1.00e-03
  [F1Callback] Epoch 008 | val_macro_f1=0.5824  ✓ Saved best weights
  [LRMonitor]  Epoch 008 | lr = 1.00e-03
  [F1Callback] Epoch 009 | val_macro_f1=0.5622
  [LRMonitor]  Epoch 009 | lr = 1.00e-03
  [F1Callback] Epoch 010 | val_macro_f1=0.7348  ✓ Saved best weights
  [LRMonitor]  Epoch 010 | lr = 1.00e-03
  [F1Callb

In [11]:
all_results = ml_results + [dl_result]
final_evaluation(all_results, data["y_test"])

inference_example(
    fitted_ml_models, dl_model,
    data["scaler"], data["feature_names"]
)

print("\n Pipeline complete. All artifacts saved to outputs/")

FINAL EVALUATION SUMMARY

             Model  Val ROC-AUC  Test ROC-AUC  Test PR-AUC
          XGBoost     0.927203      0.835249     0.571741
    Deep Learning     0.908046      0.819923     0.574918
Gradient Boosting     0.831418      0.812261     0.550792
    Random Forest     0.919540      0.785441     0.428194
         LightGBM     0.808429      0.747126     0.429715
INFERENCE SIM

New sample features:
 Age  Education  Initial_Capital  Financial_Record_Keeping  Internet_Usage  Business_Plan  Marketing_Effort  Partnership  Parent_Business_Experience  Industry_Experience  Owner_Gender  Professional_Advice
  35          3                1                         1               1              1                 6            1                           1                    8             0                    5

ML Model Predictions:
  Random Forest            :  SUCCESS  (P(success) = 0.8720)
  XGBoost                  :  SUCCESS  (P(success) = 0.9985)
  LightGBM                 :  SUCC

# Export

In [12]:
import joblib

joblib.dump(fitted_ml_models['Random Forest'], "random_forest.pkl")

joblib.dump(fitted_ml_models['Gradient Boosting'], "gradient_boosting.pkl")

fitted_ml_models['XGBoost'].save_model('xgboost.json')

fitted_ml_models['LightGBM'].booster_.save_model('lightgbm.txt')

dl_model.save('dl_classifier.keras')